In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，加载数据和CV设置
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# 创建输出文件夹
Path('models/ridge').mkdir(parents=True, exist_ok=True)

# 加载数据
df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)
with open('data/cv_setup_inner.pkl', 'rb') as f:
    cv_setup_inner = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']
outer_folds  = cv_setup['outer_folds']

X = df[feature_cols].values
y = df[target_col].values

# 超参数搜索空间
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}

# 边缘原子和原子类型配置
EDGE_THRESHOLD   = 0.1
ATOM_TYPE_COLS   = ['T_O', 'T_C', 'T_Ti1', 'T_Ti2']
ATOM_TYPE_LABELS = ['O', 'C', 'Ti_inner', 'Ti_outer']




In [2]:
# =============================================================================
# 辅助函数：计算指标，分类评估
# =============================================================================

def calculate_metrics(y_true, y_pred):
    return {
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'rmse':      float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2':        float(r2_score(y_true, y_pred)),
        'n_samples': int(len(y_true))
    }

def classify_and_evaluate(y_true, y_pred, df_subset):
    results = {}
    results['overall'] = calculate_metrics(y_true, y_pred)

    edge_mask     = df_subset['D_E'].values < EDGE_THRESHOLD
    interior_mask = ~edge_mask

    results['edge']     = calculate_metrics(y_true[edge_mask],     y_pred[edge_mask])     if edge_mask.sum()     > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    results['interior'] = calculate_metrics(y_true[interior_mask], y_pred[interior_mask]) if interior_mask.sum() > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    results['by_atom_type'] = {}
    for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
        mask = df_subset[col].values == 1
        results['by_atom_type'][label] = calculate_metrics(y_true[mask], y_pred[mask]) if mask.sum() > 0 \
            else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    return results




In [3]:
# =============================================================================
# 嵌套交叉验证主循环
# =============================================================================

outer_results          = []
best_models            = []
all_train_predictions  = []
all_test_predictions   = []
classification_results = {'fold_wise': {}, 'averaged': {}}

start_time = time.time()

for fold_idx, outer_fold in enumerate(outer_folds):
    train_idx = outer_fold['train_atoms_idx']
    test_idx  = outer_fold['test_atoms_idx']

    X_train = X[train_idx];  y_train = y[train_idx]
    X_test  = X[test_idx];   y_test  = y[test_idx]

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # 构建内层CV索引
    inner_folds = cv_setup_inner[fold_idx]['inner_folds']
    inner_cv_indices = []
    for inner_fold in inner_folds:
        inner_train_set = set(inner_fold['inner_train_structures'])
        inner_val_set   = set(inner_fold['inner_val_structures'])
        struct_ids = df.iloc[train_idx]['structure_id'].values
        inner_cv_indices.append((
            np.where([s in inner_train_set for s in struct_ids])[0],
            np.where([s in inner_val_set   for s in struct_ids])[0]
        ))

    # 超参数搜索
    grid_search = GridSearchCV(
        estimator=Ridge(),
        param_grid=param_grid,
        cv=inner_cv_indices,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train_scaled, y_train)
    best_params = grid_search.best_params_

    # 用最优参数训练
    best_model = Ridge(**best_params)
    best_model.fit(X_train_scaled, y_train)

    y_train_pred = best_model.predict(X_train_scaled)
    y_test_pred  = best_model.predict(X_test_scaled)

    train_class_results = classify_and_evaluate(y_train, y_train_pred, df_train)
    test_class_results  = classify_and_evaluate(y_test,  y_test_pred,  df_test)

    print(f"折叠 {fold_idx+1} | 最优参数: {best_params} | "
          f"训练MAE: {train_class_results['overall']['mae']:.5f} | "
          f"测试MAE: {test_class_results['overall']['mae']:.5f}")

    # 保存折叠结果
    fold_result = {
        'fold': fold_idx + 1,
        'best_params': best_params,
        'train_metrics': train_class_results,
        'test_metrics':  test_class_results,
        'n_train': len(train_idx),
        'n_test':  len(test_idx),
        'train_structures': outer_fold['train_structures'],
        'test_structures':  outer_fold['test_structures']
    }
    outer_results.append(fold_result)
    best_models.append({'fold': fold_idx + 1, 'model': best_model,
                        'scaler': scaler, 'best_params': best_params})

    classification_results['fold_wise'][f'fold_{fold_idx+1}'] = {
        'train': train_class_results,
        'test':  test_class_results
    }

    # 保存预测记录
    for i in range(len(train_idx)):
        all_train_predictions.append({
            'fold': fold_idx + 1,
            'atom_index': int(train_idx[i]),
            'structure_id': df_train.iloc[i]['structure_id'],
            'D_E': float(df_train.iloc[i]['D_E']),
            'T_O': int(df_train.iloc[i]['T_O']),
            'T_C': int(df_train.iloc[i]['T_C']),
            'T_Ti1': int(df_train.iloc[i]['T_Ti1']),
            'T_Ti2': int(df_train.iloc[i]['T_Ti2']),
            'y_true': float(y_train[i]),
            'y_pred': float(y_train_pred[i]),
            'error': float(y_train_pred[i] - y_train[i]),
            'abs_error': float(abs(y_train_pred[i] - y_train[i]))
        })
    for i in range(len(test_idx)):
        all_test_predictions.append({
            'fold': fold_idx + 1,
            'atom_index': int(test_idx[i]),
            'structure_id': df_test.iloc[i]['structure_id'],
            'D_E': float(df_test.iloc[i]['D_E']),
            'T_O': int(df_test.iloc[i]['T_O']),
            'T_C': int(df_test.iloc[i]['T_C']),
            'T_Ti1': int(df_test.iloc[i]['T_Ti1']),
            'T_Ti2': int(df_test.iloc[i]['T_Ti2']),
            'y_true': float(y_test[i]),
            'y_pred': float(y_test_pred[i]),
            'error': float(y_test_pred[i] - y_test[i]),
            'abs_error': float(abs(y_test_pred[i] - y_test[i]))
        })

elapsed_time = time.time() - start_time
print(f"\n嵌套CV完成，耗时 {elapsed_time:.1f} 秒")




折叠 1 | 最优参数: {'alpha': 0.1} | 训练MAE: 0.01155 | 测试MAE: 0.01242
折叠 2 | 最优参数: {'alpha': 0.1} | 训练MAE: 0.01175 | 测试MAE: 0.01121
折叠 3 | 最优参数: {'alpha': 0.1} | 训练MAE: 0.01185 | 测试MAE: 0.01162
折叠 4 | 最优参数: {'alpha': 0.001} | 训练MAE: 0.01165 | 测试MAE: 0.01185
折叠 5 | 最优参数: {'alpha': 0.1} | 训练MAE: 0.01177 | 测试MAE: 0.01171

嵌套CV完成，耗时 67.5 秒


In [4]:
# =============================================================================
# 计算平均性能，训练最终模型，保存所有结果
# =============================================================================

def calculate_averaged_metrics(outer_results, dataset_type='test'):
    key = f'{dataset_type}_metrics'
    averaged = {}

    for category in ['overall', 'edge', 'interior']:
        maes  = [r[key][category]['mae']  for r in outer_results]
        rmses = [r[key][category]['rmse'] for r in outer_results]
        r2s   = [r[key][category]['r2']   for r in outer_results]
        ns    = [r[key][category]['n_samples'] for r in outer_results]
        averaged[category] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }

    averaged['by_atom_type'] = {}
    for label in ATOM_TYPE_LABELS:
        maes  = [r[key]['by_atom_type'][label]['mae']  for r in outer_results]
        rmses = [r[key]['by_atom_type'][label]['rmse'] for r in outer_results]
        r2s   = [r[key]['by_atom_type'][label]['r2']   for r in outer_results]
        ns    = [r[key]['by_atom_type'][label]['n_samples'] for r in outer_results]
        averaged['by_atom_type'][label] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }

    return averaged

train_averaged = calculate_averaged_metrics(outer_results, 'train')
test_averaged  = calculate_averaged_metrics(outer_results, 'test')
classification_results['averaged'] = {'train': train_averaged, 'test': test_averaged}

# 打印平均性能
print("\n5折CV平均性能（测试集）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")
print("  按原子类型:")
for label in ATOM_TYPE_LABELS:
    m = test_averaged['by_atom_type'][label]
    print(f"    {label:12s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}")

# 最终模型（使用最常见的最优alpha）
best_alphas = [r['best_params']['alpha'] for r in outer_results]
final_alpha = max(set(best_alphas), key=best_alphas.count)
print(f"\n最终模型 alpha: {final_alpha}  (各折最优alpha: {best_alphas})")

scaler_final   = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)
final_model    = Ridge(alpha=final_alpha)
final_model.fit(X_scaled_final, y)

# 保存预测CSV（每折分别保存）
df_train_pred_all = pd.DataFrame(all_train_predictions)
df_test_pred_all  = pd.DataFrame(all_test_predictions)

for fold_num in range(1, 6):
    df_train_pred_all[df_train_pred_all['fold'] == fold_num].to_csv(
        f'models/ridge/fold_{fold_num}_train_predictions.csv', index=False)
    df_test_pred_all[df_test_pred_all['fold'] == fold_num].to_csv(
        f'models/ridge/fold_{fold_num}_test_predictions.csv', index=False)

# 保存分类评分CSV
classification_rows = []
for fold_num in range(1, 6):
    fold_data = classification_results['fold_wise'][f'fold_{fold_num}']
    for dataset in ['train', 'test']:
        for category in ['overall', 'edge', 'interior']:
            d = fold_data[dataset][category]
            classification_rows.append({
                'fold': fold_num, 'dataset': dataset,
                'category': category, 'subcategory': 'all',
                'mae': d['mae'], 'rmse': d['rmse'],
                'r2': d['r2'], 'n_samples': d['n_samples']
            })
        for label in ATOM_TYPE_LABELS:
            d = fold_data[dataset]['by_atom_type'][label]
            classification_rows.append({
                'fold': fold_num, 'dataset': dataset,
                'category': 'atom_type', 'subcategory': label,
                'mae': d['mae'], 'rmse': d['rmse'],
                'r2': d['r2'], 'n_samples': d['n_samples']
            })

pd.DataFrame(classification_rows).to_csv(
    'models/ridge/classification_scores.csv', index=False)

# 保存模型文件
model_package = {
    'model': final_model, 'scaler': scaler_final,
    'feature_cols': feature_cols, 'target_col': target_col,
    'best_params': {'alpha': final_alpha},
    'cv_performance': {'train': train_averaged, 'test': test_averaged},
    'edge_threshold': EDGE_THRESHOLD,
    'atom_type_labels': ATOM_TYPE_LABELS
}
with open('models/ridge/ridge_final_model.pkl', 'wb') as f:
    pickle.dump(model_package, f)

with open('models/ridge/ridge_all_folds.pkl', 'wb') as f:
    pickle.dump(best_models, f)

results_dict = {
    'model_name': 'Ridge Regression',
    'n_folds': len(outer_folds),
    'n_features': len(feature_cols),
    'training_time_seconds': float(elapsed_time),
    'edge_threshold': EDGE_THRESHOLD,
    'param_grid': {k: [float(v) for v in vals] for k, vals in param_grid.items()},
    'cv_results': {'train': train_averaged, 'test': test_averaged},
    'fold_details': outer_results,
    'final_model': {'alpha': float(final_alpha)}
}
with open('models/ridge/ridge_results.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print("\n已保存: 预测CSV, classification_scores.csv, "
      "ridge_final_model.pkl, ridge_all_folds.pkl, ridge_results.json")


5折CV平均性能（测试集）:
  overall    - MAE: 0.01176 ± 0.00039  RMSE: 0.02391 ± 0.00123  R²: 0.999121 ± 0.000091
  edge       - MAE: 0.05138 ± 0.00278  RMSE: 0.06698 ± 0.00364  R²: 0.990720 ± 0.000932
  interior   - MAE: 0.00706 ± 0.00032  RMSE: 0.01034 ± 0.00022  R²: 0.999841 ± 0.000007
  按原子类型:
    O            - MAE: 0.00958 ± 0.00025
    C            - MAE: 0.00806 ± 0.00061
    Ti_inner     - MAE: 0.01639 ± 0.00064
    Ti_outer     - MAE: 0.01534 ± 0.00106

最终模型 alpha: 0.1  (各折最优alpha: [0.1, 0.1, 0.1, 0.001, 0.1])

已保存: 预测CSV, classification_scores.csv, ridge_final_model.pkl, ridge_all_folds.pkl, ridge_results.json
